# Multi-store location verification

The original AI verification used one representative FHRS establishment for each unique cleaned business name. For businesses with multiple FHRS locations, this could hide differences between individual establishments.

This notebook therefore expands the multi-store bakery candidates back to their individual FHRS locations and verifies the additional locations separately. The representative location already checked previously is removed from the new verification queue.

In [15]:
from pathlib import Path
from getpass import getpass
from datetime import datetime

import json
import re
import time
import pandas as pd

from google import genai
from google.genai import types, errors


# Verification settings
MODEL = "gemini-2.5-flash"

BATCH_SIZE = 10
MAX_REQUESTS_PER_RUN = 100
REQUEST_DELAY = 2

# Paths
FHRS_DATE = "2026-07-23"

RAW_FOLDER = Path("../data/business/raw")
BUSINESS_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = BUSINESS_FOLDER / "ai_verification_v3"

AI_RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results.csv")
FHRS_PATH = (BUSINESS_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATE}.csv")
RAW_FHRS_PATH = (RAW_FOLDER / f"london_fhrs_raw_{FHRS_DATE}.csv")

OUTPUT_FOLDER = (VERIFICATION_FOLDER / "multistore_location_verification")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = (OUTPUT_FOLDER / "bakery_multistore_location_results.csv")
BATCH_LOG_PATH = (OUTPUT_FOLDER / "bakery_multistore_location_batches.jsonl")
ERROR_LOG_PATH = (OUTPUT_FOLDER / "bakery_multistore_location_errors.jsonl")

VERIFICATION_PATH = (OUTPUT_FOLDER / "bakery_multistore_location_verified.csv")

In [16]:
ai_results = pd.read_csv(AI_RESULTS_PATH, low_memory=False)

required_ai_columns = {"BusinessNameClean",
                       "FHRSIDRep",
                       "AIClass",
                       "StoreCount",
                       "VerificationDateTime"}

for column in required_ai_columns:
    if column not in ai_results.columns:
        raise ValueError(f"AI results are missing the required column: {column}")

ai_results = (ai_results
              .sort_values("VerificationDateTime")
              .drop_duplicates(subset="BusinessNameClean", keep="last")
              .reset_index(drop=True))

ai_results["StoreCount"] = (pd.to_numeric(ai_results["StoreCount"], errors="coerce")
                            .fillna(0)
                            .astype(int))

multi_store_names = (ai_results[ai_results["StoreCount"] > 1].copy())

print(f"Candidate total: {len(ai_results)}")
print(f"Multi-store candidates total: {len(multi_store_names)}")
print(f"Locations represented: {multi_store_names['StoreCount'].sum()}")

Candidate total: 28847
Multi-store candidates total: 1888
Locations represented: 8340


## Creating the location-level queue

The multi-store candidate names are matched back to the individual FHRS establishments. The representative establishment used during the original AI verification is then removed, leaving only locations that have not yet been checked individually.

In [26]:
fhrs = pd.read_csv(FHRS_PATH, low_memory=False)

required_fhrs_columns = {"FHRSID",
                         "BusinessNameClean",
                         "BusinessName",
                         "BusinessType",
                         "PostCode",
                         "LocalAuthorityName",
                         "BakeryRank",
                         "BakeryScore",
                         "StoreCount"}

for column in required_fhrs_columns:
    if column not in fhrs.columns:
        raise ValueError(f"Ranked FHRS data are missing the required column: {column}")

# Make the two FHRSID columns directly comparable
fhrs["FHRSID"] = (pd.to_numeric(fhrs["FHRSID"],  errors="coerce")
                  .astype("Int64"))

multi_store_names["FHRSIDRep"] = (pd.to_numeric(multi_store_names["FHRSIDRep"], errors="coerce")
                                  .astype("Int64"))


if fhrs["FHRSID"].isna().any():
    raise ValueError("Some ranked FHRS rows have missing or invalid FHRSIDs.")

if multi_store_names["FHRSIDRep"].isna().any():
    raise ValueError("Some representative results have missing or invalid FHRSIDs.")

duplicate_fhrsids = (fhrs["FHRSID"].duplicated().sum())

if duplicate_fhrsids:
    raise ValueError(
        "Ranked FHRS data contain "
        f"{duplicate_fhrsids} duplicate FHRSIDs."
    )

# Retain the representative result for later comparison
name_info = (multi_store_names[["BusinessNameClean",
                                "FHRSIDRep",
                                "AIClass"]]
                                .rename(columns={"AIClass": "RepresentativeClass"}))

queue = fhrs.merge(name_info,
                   on="BusinessNameClean",
                   how="inner",
                   validate="many_to_one")

# Remove the representative locations because notebook 08 checked them
queue = (queue[queue["FHRSID"] != queue["FHRSIDRep"]].copy())

queue = (queue
         .sort_values(["BakeryRank", "FHRSID"], na_position="last")
         .reset_index(drop=True))

expected_additional_locations = int((multi_store_names["StoreCount"] - 1).sum())

print(f"Expected additional locations: {expected_additional_locations}")
print(f"Matched additional locations: {len(queue)}")

if len(queue) != expected_additional_locations:
    print("Matched location count differs from the StoreCount expectation. Check whether the ranked FHRS file has changed.")

Expected additional locations: 6452
Matched additional locations: 6452


In [27]:
address_columns = ["FHRSID",
                   "AddressLine1",
                   "AddressLine2",
                   "AddressLine3",
                   "AddressLine4"]

addresses = pd.read_csv(RAW_FHRS_PATH, usecols=address_columns, low_memory=False)

addresses["FHRSID"] = (pd.to_numeric(addresses["FHRSID"], errors="coerce")
                       .astype("Int64"))

addresses = (addresses
             .dropna(subset=["FHRSID"])
             .drop_duplicates(subset="FHRSID", keep="first"))

addresses["Address"] = (addresses[["AddressLine1",
                                   "AddressLine2",
                                   "AddressLine3",
                                   "AddressLine4"]
                                   ].apply(lambda row: ", ".join(
                                       str(value).strip()
                                       for value in row
                                       if (pd.notna(value) and str(value).strip())),
                                       axis=1))

queue = queue.drop(columns=["Address"], errors="ignore")

queue = queue.merge(addresses[["FHRSID", "Address"]],
                    on="FHRSID",
                    how="left",
                    validate="one_to_one")

missing_addresses = (queue["Address"]
                     .fillna("")
                     .eq("")
                     .sum())

print(f"Locations with no assembled address: {missing_addresses}")

Locations with no assembled address: 141


In [28]:
if RESULTS_PATH.exists():
    previous_results = pd.read_csv(RESULTS_PATH, low_memory=False)

    completed_ids = set(pd.to_numeric(previous_results["FHRSID"], errors="coerce")
                        .astype("Int64")
                        .astype("string")
                        .dropna())

else:
    completed_ids = set()

remaining_queue = (queue[~queue["FHRSID"]
                         .astype("string")
                         .isin(completed_ids)].copy().reset_index(drop=True))

max_businesses_per_run = (BATCH_SIZE * MAX_REQUESTS_PER_RUN)

run_queue = (remaining_queue
             .head(max_businesses_per_run)
             .copy()
             .reset_index(drop=True))

requests_planned = (len(run_queue) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Total additional locations: {len(queue)}")
print(f"Previously completed FHRSIDs: {len(completed_ids)}")
print(f"Remaining locations: {len(remaining_queue)}")
print(f"Locations selected this run: {len(run_queue)}")
print(f"API requests planned this run: {requests_planned}")

Total additional locations: 6452
Previously completed FHRSIDs: 6452
Remaining locations: 0
Locations selected this run: 0
API requests planned this run: 0


In [6]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

In [20]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)

def build_verification_prompt(batch):

    batch = batch.reset_index(drop=True)

    business_blocks = []

    for i, row in batch.iterrows():
        business_blocks.append(
            f"""
BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
Address: {clean_value(row["Address"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
FHRS type: {clean_value(row["BusinessType"])}
""".strip()
        )

    business_text = "\n\n".join(business_blocks)

    prompt = f"""
ROLE
    
You are classifying London food establishments for a study of bakery
provision.

Use Google Search to investigate the SPECIFIC supplied FHRS
establishment.

This is an individual location belonging to a repeated or multi-store
business name. Evidence about another branch does not establish the
operating model or bakery provision at this location.

Assign exactly one class:

CORE_BAKERY
BAKERY_CAFE
GROCER_BAKERY
NON_BAKERY
UNCLEAR


==================================================
1. EVIDENCE STANDARD
==================================================

A positive bakery classification requires AFFIRMATIVE EVIDENCE.

Do not classify a business as CORE_BAKERY, BAKERY_CAFE or
GROCER_BAKERY merely because:

- its name contains "bakery", "patisserie", "bakes", etc.;
- FHRS gives it a particular business type;
- it sells or makes some baked food;
- another branch of the same brand is a bakery;
- bakery activity seems plausible.

The evidence must support the required characteristics of the
specific supplied establishment.

IMPORTANT:

"Absence of evidence is not evidence of absence" applies when deciding
between NON_BAKERY and UNCLEAR.

It does NOT justify a positive bakery classification.

Before assigning a positive class, verify the required positive
criteria below.


==================================================
2. IDENTIFY THE ESTABLISHMENT
==================================================

Use the name, address, postcode, local authority and current web
evidence together.

A matching address/postcode is strong evidence. Where location
information is incomplete, a distinctive business in the same local
area may still be a reasonable match if there is no conflicting
evidence.

Classify the supplied establishment, NOT the brand generally.

A company's production kitchen, warehouse or office does not become
a bakery retail location merely because the company operates bakery
shops elsewhere.

FHRS BusinessType is contextual information only and must not decide
the class by itself.


==================================================
3. HARD EXCLUSIONS
==================================================

If reliable evidence establishes any of the following, classify
NON_BAKERY regardless of the business name:

- permanently closed or inactive;
- home-based only;
- online or delivery-only;
- wholesale or production-only with no customer-facing retail;
- event/catering operation with no qualifying fixed retail site;
- travelling or changing-location market stall;
- roaming/mobile operation without a stable customer-facing location.

A fixed kiosk, permanent market unit, fixed counter, or permanently
stationed food truck/trailer CAN count as a stable retail location.

For a closed business, begin the reason with:

INACTIVE:


==================================================
4. CORE_BAKERY
==================================================

CORE_BAKERY is the strict principal bakery definition.

Assign CORE_BAKERY only when BOTH are supported:

A. The supplied establishment operates from a stable,
   customer-facing retail location.

B. Its principal identity or specialism is genuinely bakery-led.

Examples include:

- bread bakeries;
- artisan/sourdough bakeries;
- bakehouses;
- boulangeries;
- genuine patisseries;
- bakery chains;
- culturally specific bakeries;
- specialist bread businesses centred on naan, roti, simit or
  similar breads.

Do not require a British or European bakery style.

A specialist naan or roti business can qualify where bread production
and sale are essentially the business itself.

A restaurant/takeaway selling naan, roti, pide or bread alongside a
wider meal menu is NON_BAKERY.


SINGLE-PRODUCT RULE

A specialist snack/dessert concept is not CORE_BAKERY simply because
its product is baked or dough-based.

Normally NON_BAKERY:

- bespoke/celebration-cake-only businesses;
- cookie-only businesses;
- doughnut-only businesses;
- pretzel-only businesses;
- cinnamon-roll-only businesses;
- macaron-only businesses;
- churro-only businesses;
- waffle/crepe-only businesses;
- similar narrow snack or dessert concepts.

A specialist bread bakery or genuine patisserie with a recognisable
bakery/pastry range can still qualify.


==================================================
5. BAKERY_CAFE
==================================================

Assign BAKERY_CAFE only when BOTH are supported:

A. The establishment operates as a cafe, sandwich shop or similar
   customer-facing food-service business.

B. Current menu/product evidence shows that bakery goods form a
   SUBSTANTIAL and PERSISTENT part of what customers can buy.

This is deliberately broader than CORE_BAKERY.

A business does not need to be primarily a bakery.

Examples can include a cafe with a substantial ongoing range of
breads, pastries, cakes and related bakery goods.

One or two pastries, some cakes, bread with meals, pizza, naan, pide
or incidental desserts are NOT enough.

When uncertain, inspect a current menu, ordering page or product range
before assigning BAKERY_CAFE.


==================================================
6. GROCER_BAKERY
==================================================

Assign GROCER_BAKERY only when BOTH are supported:

A. The establishment is a supermarket, grocer, deli, convenience
   store or other general food retailer.

B. There is AFFIRMATIVE evidence of substantial bakery provision at
   that supplied location.

Qualifying evidence can include:

- a dedicated bakery section;
- an in-store bakery;
- an identifiable bakery concession;
- a broad persistent fresh bakery range;
- bakery products forming a substantial part of the retail offering.

Examples:

Supermarket with a substantial in-store bakery
-> GROCER_BAKERY

Independent grocer with a large fresh bread/pastry operation
-> GROCER_BAKERY

Deli with a substantial persistent bakery range
-> GROCER_BAKERY

Asda containing an operating Greggs concession
-> GROCER_BAKERY

The following evidence is NOT enough by itself:

- sells bread;
- sells pastries;
- sells baked goods;
- may contain baked goods;
- stocks packaged cakes or bread;
- a generic chain website says some branches have bakeries.

The bakery evidence must apply to the supplied location and be
substantial rather than incidental.


==================================================
7. NON_BAKERY AND UNCLEAR
==================================================

NON_BAKERY:

Use when the establishment has been reasonably identified and the
evidence shows that it does not satisfy any positive category.

Typical examples include:

- ordinary restaurants;
- pizzerias;
- pubs/hotels;
- restaurants making bread as part of a wider menu;
- cafes with only minor bakery offerings;
- retailers with only incidental bakery products;
- the single-product snack/dessert concepts listed above;
- any hard exclusion listed earlier.


UNCLEAR:

Use only where the relevant establishment, location or operating model
genuinely cannot be established from available evidence.

If a business appears bakery-like but the evidence does not establish
the required customer-facing operation, use UNCLEAR rather than
inventing positive evidence.

Do not use UNCLEAR merely because the distinction between two known
categories requires judgement.


==================================================
8. DECISION ORDER
==================================================

For each business:

1. Can the supplied establishment reasonably be identified?
   No -> UNCLEAR

2. Does a hard exclusion apply?
   Yes -> NON_BAKERY

3. Are BOTH CORE_BAKERY requirements affirmatively supported?
   Yes -> CORE_BAKERY

4. Are BOTH BAKERY_CAFE requirements affirmatively supported?
   Yes -> BAKERY_CAFE

5. Are BOTH GROCER_BAKERY requirements affirmatively supported?
   Yes -> GROCER_BAKERY

6. Otherwise:
   -> NON_BAKERY if evidence establishes the operating model
   -> UNCLEAR if the relevant operating model/location remains
      genuinely unresolved


==================================================
9. SOURCES
==================================================

Prefer current evidence such as:

- official business website;
- official menu/order page;
- official social media;
- current delivery platform;
- market/shopping-centre/operator website;
- credible current directories.

For positive classifications, actively look for evidence of BOTH the
business operation and the bakery offering.

Do not use the business name alone as evidence.


==================================================
10. OUTPUT
==================================================

Return exactly {len(batch)} lines.

Every line must contain exactly four pipe-separated fields:

business number | location match | classification | short reason

Location match:

YES
UNCLEAR

If location match is UNCLEAR, class must be UNCLEAR.

Examples:

1 | YES | CORE_BAKERY | Fixed specialist bread bakery with customer-facing retail at the supplied location.
2 | YES | BAKERY_CAFE | Cafe whose current menu shows a substantial persistent bakery range.
3 | YES | GROCER_BAKERY | Supermarket with a verified dedicated in-store bakery at this location.
4 | YES | NON_BAKERY | Restaurant sells flatbread but bakery products are incidental to its wider menu.
5 | UNCLEAR | UNCLEAR | Available evidence cannot establish the operation at the supplied location.
6 | YES | NON_BAKERY | INACTIVE: The supplied establishment has permanently closed.

Return every business number exactly once.

Return ONLY the result lines.


==================================================
BUSINESSES
==================================================

{business_text}
""".strip()

    return prompt

In [21]:
def verify_current_batch(batch):

    prompt = build_verification_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0,
            max_output_tokens=1500,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            )
        )
    )

    return response

In [22]:
def get_grounding_info(response):

    if not response.candidates:
        return False, [], [], 0

    grounding = response.candidates[0].grounding_metadata

    if grounding is None:
        return False, [], [], 0

    search_queries = list(grounding.web_search_queries or [])

    source_urls = []

    for chunk in grounding.grounding_chunks or []:
        if chunk.web:
            source_urls.append(chunk.web.uri)

    source_urls = list(dict.fromkeys(source_urls))

    support_count = len(grounding.grounding_supports or [])

    grounding_used = bool(search_queries)

    return (grounding_used, search_queries, source_urls, support_count)

In [23]:
def parse_response(response_text, batch, batch_id):

    batch = (batch.reset_index(drop=True))

    text = (str(response_text or "")
            .replace("\\_", "_")
            .strip())

    # Add a newline in case Gemini joins consecutive results
    text = re.sub((r"(?<!^)(?<!\n)(?<!\d)"
                   r"(?=(?:\*{0,2})?\d{1,2}\s*\|)"), "\n", text)

    allowed_classes = {"CORE_BAKERY",
                       "BAKERY_CAFE",
                       "GROCER_BAKERY",
                       "NON_BAKERY",
                       "UNCLEAR"}

    parsed = {}

    verification_time = (datetime.now().isoformat(timespec="seconds"))

    for line in text.splitlines():
        parts = [part.strip() for part in line.split("|", 3)]

        if len(parts) != 4:
            continue

        number_text = (parts[0]
                       .replace("*", "")
                       .strip())

        if not number_text.isdigit():
            continue

        business_number = int(number_text)

        if not 1 <= business_number <= len(batch):
            continue

        # Accept each requested number once only
        if business_number in parsed:
            continue

        location_match = (parts[1].upper())

        classification = (parts[2]
                          .upper()
                          .replace(" ", "_"))

        reason = parts[3].strip()

        if location_match not in {"YES", "UNCLEAR"}:
            continue

        if classification not in allowed_classes:
            continue

        if (location_match == "UNCLEAR" and classification != "UNCLEAR"):
            continue

        if not reason:
            continue

        row = batch.iloc[business_number - 1]

        parsed[business_number] = {"FHRSID": row["FHRSID"],
                                   "BusinessNameClean": (row["BusinessNameClean"]),
                                   "BusinessName": (row["BusinessName"]),
                                   "FHRSIDRep": row["FHRSIDRep"],
                                   "RepresentativeClass": (row["RepresentativeClass"]),
                                   "BusinessType": (row["BusinessType"]),
                                   "PostCode": row["PostCode"],
                                   "LocalAuthorityName": (row["LocalAuthorityName"]),
                                   "Address": row["Address"],
                                   "BakeryRank": (row["BakeryRank"]),
                                   "BakeryScore": (row["BakeryScore"]),
                                   "StoreCount": (row["StoreCount"]),
                                   "LocationMatch": (location_match),
                                   "AIClass": classification,
                                   "AIReason": reason,
                                   "BatchID": batch_id,
                                   "Model": MODEL,
                                   "VerificationDateTime": (verification_time)}

    missing_numbers = [number
                       for number in range(1, len(batch) + 1)
                       if number not in parsed]

    results = pd.DataFrame([parsed[number]
                            for number in sorted(parsed)])

    return results, missing_numbers


def save_results(results):

    if results.empty:
        return 0

    result_ids = (results["FHRSID"].astype("string"))

    # Remove locations already completed in earlier batches or notebooks
    results = (results[~result_ids.isin(completed_ids)].copy())

    if results.empty:
        return 0

    results.to_csv(RESULTS_PATH, 
                   mode="a",
                   header=not RESULTS_PATH.exists(),
                   index=False)

    completed_ids.update(results["FHRSID"].astype("string"))

    return len(results)


def save_jsonl(path, record):
    with path.open("a", encoding="utf-8") as file:

        file.write(json.dumps(record, ensure_ascii=False, default=str)+ "\n")

In [24]:
RUN_ID = (datetime.now().strftime("%Y%m%d_%H%M%S"))

requests_attempted = 0
results_saved = 0

for start in range(0, len(run_queue), BATCH_SIZE):
    batch = (run_queue.iloc[start:start + BATCH_SIZE]
             .copy()
             .reset_index(drop=True))

    requests_attempted += 1

    batch_id = (f"{RUN_ID}_{requests_attempted}")

    try:
        response = (verify_current_batch(batch))

        response_text = (response.text or "")

        finish_reason = None
        finish_message = None

        if response.candidates:
            finish_reason = str(response.candidates[0].finish_reason)

            finish_message = (response.candidates[0].finish_message)

        (grounding_used, search_queries, source_urls,support_count) = get_grounding_info(response)

        batch_record = {"BatchID": batch_id,
                        "Time": (datetime.now().isoformat(timespec="seconds")),
                        "BusinessCount": len(batch),
                        "FHRSIDs": (batch["FHRSID"].astype(str).tolist()),
                        "BusinessNames": (batch["BusinessName"].tolist()),
                        "RawResponse": response_text,
                        "GroundingUsed": (grounding_used),
                        "SearchQueries": (search_queries),
                        "SourceURLs": source_urls,
                        "GroundingSupportCount": (support_count),
                        "Model": MODEL,
                        "FinishReason": (finish_reason),
                        "FinishMessage": (finish_message)}

        # Do not accept ungrounded classifications
        if not grounding_used:

            batch_record["ParsedCount"] = 0

            save_jsonl(BATCH_LOG_PATH, batch_record)

            save_jsonl(ERROR_LOG_PATH,
                       {"BatchID": batch_id,
                        "ErrorType": ("NO_SEARCH_GROUNDING"),
                        "FHRSIDs": (batch["FHRSID"].astype(str).tolist()),
                        "Businesses": (batch["BusinessName"].tolist())})

            print(f"Batch {requests_attempted}: no Google Search grounding - not saved")

            if (start + BATCH_SIZE < len(run_queue)):
                time.sleep(REQUEST_DELAY)

            continue

        results, missing_numbers = (parse_response(response_text, batch, batch_id))

        batch_record["ParsedCount"] = (len(results))
        batch_record["MissingNumbers"] = (missing_numbers)

        save_jsonl(BATCH_LOG_PATH, batch_record)

        saved_count = save_results(results)
        results_saved += saved_count

        if missing_numbers:
            missing_rows = batch.iloc[[number - 1
                                       for number in missing_numbers]]

            save_jsonl(ERROR_LOG_PATH, {"BatchID": batch_id,
                                        "ErrorType": ("INCOMPLETE_RESPONSE"),
                                        "FHRSIDs": (missing_rows["FHRSID"].astype(str).tolist()),
                                        "Businesses": (missing_rows["BusinessName"].tolist())})

        print(f"Batch {requests_attempted}: saved {saved_count}/{len(batch)}")

    except errors.APIError as error:

        error_code = getattr(error, "code", None)

        save_jsonl(ERROR_LOG_PATH,{"BatchID": batch_id,
                                   "ErrorType": "API_ERROR",
                                   "ErrorCode": error_code,
                                   "ErrorMessage": getattr(error, "message", str(error)),
                                   "FHRSIDs": (batch["FHRSID"].astype(str).tolist()),
                                   "Businesses": (batch["BusinessName"].tolist())})

        print(f"Batch {requests_attempted} failed: {error_code}")

        if error_code == 429:
            print("Rate limit reached. Stopping safely.")
            break

        if (isinstance(error_code, int) and 400 <= error_code < 500):
            print("Non-retryable API error. Stopping safely.")
            break

    except Exception as error:

        save_jsonl(ERROR_LOG_PATH, {"BatchID": batch_id,
                                    "ErrorType": (type(error).__name__),
                                    "ErrorMessage": str(error),
                                    "FHRSIDs": (batch["FHRSID"].astype(str).tolist()),
                                    "Businesses": (batch["BusinessName"].tolist())})

        print(f"Batch {requests_attempted} failed: {error}")

    if (start + BATCH_SIZE < len(run_queue)):
        time.sleep(REQUEST_DELAY)

print(f"Requests attempted this run: {requests_attempted}")

print(f"New location results saved: {results_saved}")

Batch 1: saved 10/10
Batch 2: saved 10/10
Batch 3: saved 10/10
Batch 4: saved 10/10
Batch 5: saved 10/10
Batch 6: saved 10/10
Batch 7: saved 10/10
Batch 8: saved 10/10
Batch 9: saved 10/10
Batch 10: saved 10/10
Batch 11: saved 10/10
Batch 12: saved 10/10
Batch 13: saved 10/10
Batch 14: saved 10/10
Batch 15: saved 10/10
Batch 16: saved 10/10
Batch 17: saved 10/10
Batch 18: saved 10/10
Batch 19: saved 10/10
Batch 20: saved 10/10
Batch 21: saved 10/10
Batch 22: saved 10/10
Batch 23: saved 10/10
Batch 24: saved 10/10
Batch 25: saved 10/10
Batch 26: saved 10/10
Batch 27: saved 10/10
Batch 28: saved 10/10
Batch 29: saved 10/10
Batch 30: saved 10/10
Batch 31: saved 10/10
Batch 32: saved 10/10
Batch 33: saved 10/10
Batch 34: saved 10/10
Batch 35: saved 10/10
Batch 36: saved 10/10
Batch 37: saved 10/10
Batch 38: saved 10/10
Batch 39: saved 10/10
Batch 40: saved 10/10
Batch 41: saved 8/8
Requests attempted this run: 41
New location results saved: 408


In [25]:
if RESULTS_PATH.exists():
    location_results = pd.read_csv(RESULTS_PATH, low_memory=False)

    location_results["FHRSID"] = (pd.to_numeric(location_results["FHRSID"],
                                                errors="coerce"
                                                ).astype("Int64"))

    # Keep the most recent result in case of duplicates
    location_results = (location_results
                        .sort_values("VerificationDateTime")
                        .drop_duplicates(subset="FHRSID", keep="last")
                        .reset_index(drop=True))

    missing_locations = (queue[~queue["FHRSID"].isin(location_results["FHRSID"])].copy())

    completed_locations = (len(queue) - len(missing_locations))

    print(f"Locations to verify: {len(queue)}")
    print(f"Locations completed: {completed_locations}")
    print(f"Locations still missing: {len(missing_locations)}")

    print("\nLocation classifications:")
    display(location_results["AIClass"].value_counts())

else:
    location_results = pd.DataFrame()

    missing_locations = queue.copy()

    print("No location results have been saved yet.")

Locations to verify: 6452
Locations completed: 6452
Locations still missing: 0

Location classifications:


AIClass
NON_BAKERY       4076
GROCER_BAKERY     964
BAKERY_CAFE       723
UNCLEAR           352
CORE_BAKERY       337
Name: count, dtype: int64

In [30]:
bakery_classes = ["CORE_BAKERY",
                  "BAKERY_CAFE",
                  "GROCER_BAKERY"]

comparison = location_results.copy()

representative_in = (comparison["RepresentativeClass"].isin(bakery_classes))
verified_in = (comparison["AIClass"].isin(bakery_classes))
stayed_in = (representative_in & verified_in).sum()

in_to_out = (representative_in & ~verified_in).sum()
out_to_in = (~representative_in & verified_in).sum()

stayed_out = (~representative_in & ~verified_in).sum()

net_change = (out_to_in - in_to_out)

comparison["BakeryIncluded"] = (verified_in)

print(f"Stayed in: {stayed_in}")
print(f"In to out: {in_to_out}")
print(f"Out to in: {out_to_in}")
print(f"Stayed out: {stayed_out}")
print(f"Net bakery change: {net_change}")

Stayed in: 1760
In to out: 357
Out to in: 264
Stayed out: 4071
Net bakery change: -93


# Sample the results that have the greatest effect

In [31]:
in_to_out_sample = (comparison[representative_in & ~verified_in]
                    .sample(n=10, random_state=42)
                    .copy())

in_to_out_sample["SampleGroup"] = ("IN_TO_OUT")

out_to_in_sample = (comparison[~representative_in & verified_in]
                    .sample(n=10, random_state=42)
                    .copy())

out_to_in_sample["SampleGroup"] = ("OUT_TO_IN")

stayed_in_sample = (comparison[representative_in & verified_in]
                    .sample(n=5, random_state=42)
                    .copy())

stayed_in_sample["SampleGroup"] = ("STAYED_IN")

validation_sample = pd.concat(
    [in_to_out_sample, out_to_in_sample, stayed_in_sample],
    ignore_index=True)

# Five UNCLEAR results that have not been selected
unclear_sample = (comparison[(comparison["AIClass"] == "UNCLEAR")
                             & ~comparison["FHRSID"].isin(validation_sample["FHRSID"])
                             ].sample(n=5, random_state=42)
                             .copy())

unclear_sample["SampleGroup"] = ("UNCLEAR")

validation_sample = pd.concat(
    [validation_sample, unclear_sample],
    ignore_index=True)

validation_sample = validation_sample[["FHRSID",
                                       "BusinessName",
                                       "Address",
                                       "PostCode",
                                       "RepresentativeClass",
                                       "AIClass",
                                       "AIReason",
                                       "SampleGroup"]].copy()

validation_sample["CheckResult"] = ""
validation_sample["CheckNotes"] = ""

VALIDATION_SAMPLE_PATH = (OUTPUT_FOLDER / "bakery_multistore_validation_sample.csv")

validation_sample.to_csv(VALIDATION_SAMPLE_PATH, index=False)

print(f"Saved {len(validation_sample)} locations for a quick validation check.")

print(f"Saved to: {VALIDATION_SAMPLE_PATH}")

display(validation_sample)

Saved 30 locations for a quick validation check.
Saved to: ..\data\business\interim\ai_verification_v3\multistore_location_verification\bakery_multistore_validation_sample.csv


,FHRSID,BusinessName,Address,PostCode,RepresentativeClass,AIClass,AIReason,SampleGroup,CheckResult,CheckNotes
0,694807,Iceland,"210 - 218 Trafalgar Road, Greenwich",SE10 9ER,GROCER_BAKERY,NON_BAKERY,Supermarket selling typical packaged bakery it...,IN_TO_OUT,,
1,1905201,By Anna,"Unit 3, Villiers Court, 40 Upper Mulgrave Road...",SM2 7AJ,CORE_BAKERY,UNCLEAR,Insufficient evidence to establish the operati...,IN_TO_OUT,,
2,100175,Riverside Store,"Riverside Store Unit 5 Michigan Building, 2 B...",E14 9QT,GROCER_BAKERY,NON_BAKERY,"Retailers - other, with no evidence of substan...",IN_TO_OUT,,
3,413906,The Ginger Pig,"8 MOXON STREET, LONDON",W1U 4ET,GROCER_BAKERY,NON_BAKERY,The Ginger Pig on Moxon Street is primarily a ...,IN_TO_OUT,,
4,1599346,Iceland Foods Ltd,"Iceland Foods Limited, 470 Bromley Road, London",BR1 4PP,GROCER_BAKERY,NON_BAKERY,Supermarket with no affirmative evidence of su...,IN_TO_OUT,,
5,1617849,Strawberry Local,"21 - 23 Friars Place Lane, Acton",W3 7AQ,GROCER_BAKERY,NON_BAKERY,Strawberry Local is identified as a convenienc...,IN_TO_OUT,,
6,1775239,Sweet Life,NaN,DA14,BAKERY_CAFE,UNCLEAR,"The business ""Sweet Life"" with postcode DA14 a...",IN_TO_OUT,,
7,1111366,Iceland Foods Ltd,"Iceland, 119 Cecil Road, ENFIELD",EN2 6TR,GROCER_BAKERY,NON_BAKERY,Supermarket with no affirmative evidence of su...,IN_TO_OUT,,
8,1564057,Bonne Bouche,"204 Portobello Road, LONDON",W11 1LA,BAKERY_CAFE,NON_BAKERY,This Bonne Bouche location is identified as a ...,IN_TO_OUT,,
9,654059,Shreeji News,"301 Tamworth Lane, Mitcham, Merton",CR4 1DD,GROCER_BAKERY,NON_BAKERY,Newsagent with no evidence of substantial bake...,IN_TO_OUT,,


In [33]:
validation_results = pd.read_csv(VALIDATION_SAMPLE_PATH)

display(validation_results["CheckResult"]
        .value_counts(dropna=False))

CheckResult
CORRECT    30
Name: count, dtype: int64

In [ ]:
comparison.to_csv(VERIFICATION_PATH, index=False)

print(f"Saved {len(comparison)} verified location results.")
print(f"Final bakery locations: {comparison['BakeryIncluded'].sum()}")
print(f"Saved to: {VERIFICATION_PATH}")

Saved 6452 verified location results.
Final bakery locations: 2024
Saved to: ..\data\business\interim\ai_verification_v3\multistore_location_verification\bakery_multistore_location_verified.csv
